# PdfParser Demo

Shows flat mode and tree mode extraction on `tests/fixtures/sample.pdf`.

| Aspect | Notes |
|--------|-------|
| **Pros** | Handles real-world PDFs (text, embedded images, grid tables); pymupdf is fast and low-memory; `find_tables()` detects native vector tables without regex hacks |
| **Cons** | Heading detection is heuristic (font-size ranking) — unreliable when fonts are decorative or inconsistent; multi-column layouts may be read in wrong order; no OCR for scanned-image PDFs |
| **vs. pdfplumber** | pymupdf is significantly faster and uses less memory; pdfplumber exposes more fine-grained layout data but is slower |
| **vs. unstructured.io / docling** | Those tools use ML-based layout models — more robust on complex PDFs but require heavy GPU/model dependencies; PdfParser is lighter and easier to run offline |

## Imports

In [1]:
# Standard Library
import pathlib

# Third Party Library

# Private Library
from cleave.parsers.factory import ParserFactory

## Fixture

In [2]:
FIXTURES = pathlib.Path.cwd().parent.parent / "tests" / "fixtures"
PDF_PATH = str(FIXTURES / "sample.pdf")

if not (FIXTURES / "sample.pdf").exists():
    from tests.fixtures.pdf import create_sample_pdf
    create_sample_pdf(FIXTURES / "sample.pdf")

print("PDF:", PDF_PATH)

PDF: c:\Users\SidNa\Documents\GitHub\AI\cleave\notebooks\tests\fixtures\sample.pdf


| Mode | `Document` field | Use when |
|------|-----------------|----------|
| `flat` | `.pages` — one `DocumentPage` per PDF page | Simple text retrieval, fixed chunking |
| `tree` | `.root` — recursive `TreeNode` hierarchy | Semantic / layout-aware chunking |

## Flat mode

In [3]:
flat_doc_parser = ParserFactory.create(PDF_PATH, mode="flat")
flat_doc = flat_doc_parser.parse()

print(f"Pages      : {flat_doc.total_pages}")
print(f"Root       : {flat_doc.root}")  # None in flat mode

Pages      : 1
Root       : None


In [4]:
for page in flat_doc.pages:
    print(f"--- Page {page.page_number} ({len(page.blocks)} blocks) ---")
    for block in page.blocks:
        preview = block.content[:100].replace("\n", " ")
        print(f"  [{block.type.value:5}]  {preview}")

--- Page 1 (3 blocks) ---
  [text ]  Sample Document Introduction This is the introduction paragraph. Data Overview This section covers d
  [image]  iVBORw0KGgoAAAANSUhEUgAAABQAAAAUCAIAAAAC64paAAAACXBIWXMAAA7EAAAOxAGVKw4bAAAAG0lEQVR4nGP4z8BANiJf56jm
  [table]  | Name | Value | | --- | --- | | Alpha | 1 | | Beta | 2 |


In [5]:
print(f"Tables : {len(flat_doc.all_tables)}")
print(f"Images : {len(flat_doc.all_images)}")

if flat_doc.all_tables:
    print("\nFirst table (Markdown):")
    print(flat_doc.all_tables[0].content)

Tables : 1
Images : 1

First table (Markdown):
| Name | Value |
| --- | --- |
| Alpha | 1 |
| Beta | 2 |


## Tree mode

In [6]:
tree_doc_parser = ParserFactory.create(PDF_PATH, mode="tree")
tree_doc = tree_doc_parser.parse()

print(f"Pages          : {tree_doc.pages}")  # None in tree mode
print(f"Root children  : {len(tree_doc.root.children)}")

Pages          : None
Root children  : 1


In [7]:
def print_tree(node, indent=0):
    role  = node.metadata.get("role", "")
    level = node.metadata.get("level", "")
    label = f"[{node.content_type.value}]"
    if role:  label += f" role={role}"
    if level: label += f" level={level}"
    preview = node.content[:70].replace("\n", " ") if node.content else ""
    print("  " * indent + f"{label}  \"{preview}\"")
    for child in node.children:
        print_tree(child, indent + 1)

print_tree(tree_doc.root)

[text] role=root  ""
  [text] role=heading level=1  "Sample Document"
    [text] role=heading level=2  "Introduction"
      [text] role=paragraph  "This is the introduction paragraph."
    [text] role=heading level=2  "Data Overview"
      [text] role=paragraph  "This section covers data."
      [image]  "iVBORw0KGgoAAAANSUhEUgAAABQAAAAUCAIAAAAC64paAAAACXBIWXMAAA7EAAAOxAGVKw"
      [table]  "| Name | Value | | --- | --- | | Alpha | 1 | | Beta | 2 |"
    [text] role=heading level=2  "Conclusion"
      [text] role=paragraph  "Final remarks go here."


In [8]:
print("=== FULL TEXT ===")
print(tree_doc.full_text[:400])
print("\n=== TREE TO MARKDOWN (FOR TEXT STYLE CHUNKING) ===")
print(tree_doc.to_markdown()[:400])

=== FULL TEXT ===
Sample Document
Introduction
This is the introduction paragraph.
Data Overview
This section covers data.
| Name | Value |
| --- | --- |
| Alpha | 1 |
| Beta | 2 |
Conclusion
Final remarks go here.

=== TREE TO MARKDOWN (FOR TEXT STYLE CHUNKING) ===
# Sample Document

## Introduction

This is the introduction paragraph.

## Data Overview

This section covers data.

![image_p1]()

| Name | Value |
| --- | --- |
| Alpha | 1 |
| Beta | 2 |

## Conclusion

Final remarks go here.


## Summary comparison

In [9]:
def count_nodes(node):
    return 1 + sum(count_nodes(c) for c in node.children)

print(f"{'Mode':<6}  {'Pages':>5}  {'Root nodes':>10}  {'Tables':>6}  {'Images':>6}")
print("-" * 40)
print(f"{'flat':<6}  {flat_doc.total_pages:>5}  {'N/A':>10}  {len(flat_doc.all_tables):>6}  {len(flat_doc.all_images):>6}")
print(f"{'tree':<6}  {'N/A':>5}  {count_nodes(tree_doc.root):>10}  {len(tree_doc.all_tables):>6}  {len(tree_doc.all_images):>6}")

Mode    Pages  Root nodes  Tables  Images
----------------------------------------
flat        1         N/A       1       1
tree      N/A          10       1       1
